# 04 · Визуализация

Графики и карты по файлам прогона (без DOLFINx):

1. временные ряды: потенциал, активное напряжение, растяжение волокна, напряжение;
2. карты удара: время активации с изохронами, APD, локальная скорость проведения;
3. снимки потенциала — эволюция волны;
4. механика: деформированная сетка, растяжение волокна и действующее напряжение;
5. анимация снимков;
6. APD по регионам и реституция;
7. серия: биомаркер против параметра.

Рисунки сохраняются в `runs/<имя>/figures/` (PNG, `SAVE = True`). Полные поля во времени —
в `electrics.xdmf` / `mechanics.xdmf`, их удобнее смотреть в ParaView.

In [ ]:
# Общая настройка: пакет cardiac_em и помощники ноутбуков доступны из любой папки проекта
import sys
from pathlib import Path

for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "notebooks" / "nbtools.py").exists():
        sys.path.insert(0, str(_p / "notebooks"))
        break
from nbtools import ROOT, RUNS, run_stream, style  # noqa: E402

print("корень проекта:", ROOT)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from cardiac_em.analysis import biomarkers as bm
from cardiac_em.analysis import open_run, open_sweep

style.apply()

RUN_DIR = RUNS / "nb_quick"          # ночная серия: RUNS / "night" / "run_000"
SAVE = True

run = open_run(RUN_DIR)
FIG_DIR = run.dir / "figures"
FIG_DIR.mkdir(exist_ok=True)
CELL = run.manifest["models"]["cell"]
V_NAME = "V" if CELL.startswith("tnnpm") else run.manifest["models"]["cell_states"][0]
V_UNIT = "мВ" if CELL.startswith("tnnpm") else "отн. ед."
PRELOAD = run.config["preload"]["stretch"]
E, M = run.meshes["electric"], run.meshes["mechanical"]
EXTENT = (0, E["lx_mm"], 0, E["ly_mm"])


def save(fig, name):
    if SAVE:
        fig.savefig(FIG_DIR / f"{name}.png", dpi=160, bbox_inches="tight")


def field(ax, grid, title, cmap, label, norm=None, vmin=None, vmax=None, extent=EXTENT):
    im = ax.imshow(grid, origin="lower", extent=extent, cmap=cmap, norm=norm,
                   vmin=None if norm else vmin, vmax=None if norm else vmax,
                   interpolation="bilinear", aspect="equal")
    ax.set_title(title)
    ax.set_xlabel("x, мм")
    ax.set_ylabel("y, мм")
    ax.grid(False)
    cb = plt.colorbar(im, ax=ax, shrink=0.85)
    cb.set_label(label)
    return im

print(f"прогон {run.dir.name}: модель {CELL}, снимков {len(run.snapshot_times)}")

## 1. Временные ряды

По величине на панель, общая ось времени. Где на панели две линии — легенда.

In [ ]:
s = run.series()
t = s["t_ms"]
fig, axs = plt.subplots(4, 1, figsize=(8, 9), sharex=True, constrained_layout=True)

axs[0].plot(t, s["u_max"], label="максимум")
axs[0].plot(t, s["u_min"], label="минимум")
axs[0].set_ylabel(f"{V_NAME}, {V_UNIT}")
axs[0].set_title("Потенциал по ткани")
axs[0].legend(loc="upper right")

axs[1].plot(t, s["t_act_mech_max"], label="переданное клетками (v = 0)")
axs[1].plot(t, s["t_act_actual_max"], label="действующее (с сила–скорость)")
axs[1].set_ylabel("кПа")
axs[1].set_title("Активное напряжение, максимум по ткани")
axs[1].legend(loc="upper right")

axs[2].plot(t, s["lambda_f_max"], label="максимум")
axs[2].plot(t, s["lambda_f_min"], label="минимум")
axs[2].axhline(PRELOAD, color=style.muted, lw=1, ls="--")
axs[2].annotate("преднагрузка", (t[0], PRELOAD), xytext=(4, 4), textcoords="offset points",
                ha="left", color=style.muted, fontsize=9)
axs[2].set_ylabel("λ_f")
axs[2].set_title("Растяжение волокна")
axs[2].legend(loc="lower right")

axs[3].plot(t, s["sigma_xx_max"], color=style.series[0])
axs[3].set_ylabel("кПа")
axs[3].set_title("σ_xx, максимум по ткани")
axs[3].set_xlabel("t, мс")
save(fig, "series")

## 2. Карты удара

Время активации с изохронами (линии равного времени; сгущение — замедление проведения),
APD и локальная скорость проведения |∇t_act|⁻¹. Шкала скорости — по 5–95 % внутренних узлов:
в зоне стимула градиент почти нулевой (скорость «бесконечна»), а у противоположного края
волна ускоряется на непроводящей границе — эти полосы остаются на карте, но не задают шкалу.

In [ ]:
BEAT = -1                   # удар: 0 — первый, -1 — последний
ISO_STEP_MS = None          # шаг изохрон, мс; None — ~10 линий на карту
CV_EDGE = 3                 # узлов от края, не входящих в шкалу CV (стимул, край)

try:
    maps = run.activation()
except FileNotFoundError as exc:
    maps = None
    print(exc)

B = BEAT % maps.n_beats if maps is not None and maps.n_beats else 0
if maps is not None and maps.n_beats:
    act = maps.grid(maps.act[:, B])
    apd = maps.grid(maps.apd[:, B])
    cv = bm.cv_field(act, E["lx_mm"] / E["nx"], E["ly_mm"] / E["ny"])
    inner = cv[CV_EDGE:-CV_EDGE, CV_EDGE:-CV_EDGE]
    lo, hi = np.nanpercentile(inner if np.isfinite(inner).any() else cv, [5, 95])
    span = np.nanmax(act) - np.nanmin(act)
    step = ISO_STEP_MS or float(np.round(span / 10, 1) or 0.1)

    fig, axs = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)
    field(axs[0], act, f"Время активации, удар {B + 1}", style.time, "мс")
    xs = np.linspace(0, E["lx_mm"], E["nx"] + 1)
    ys = np.linspace(0, E["ly_mm"], E["ny"] + 1)
    levels = np.arange(np.nanmin(act) + step, np.nanmax(act), step)
    cs = axs[0].contour(xs, ys, act, levels=levels, colors="white", linewidths=0.8)
    axs[0].clabel(cs, levels[::2], fontsize=7, fmt="%.1f")
    axs[0].set_title(f"Время активации, удар {B + 1} (изохроны через {step:g} мс)")
    field(axs[1], apd, f"APD{int(round(maps.apd_level * 100))}", style.seq, "мс")
    field(axs[2], np.clip(cv, lo, hi), "Локальная скорость проведения", style.seq, "мм/мс")
    print(f"CV внутри (без {CV_EDGE} узлов у краёв): медиана {np.nanmedian(inner):.2f} мм/мс; "
          f"шкала {lo:.2f}–{hi:.2f}")
    save(fig, f"beat{B + 1}_maps")

## 3. Снимки потенциала

Общая шкала для всех моментов — иначе цвет в разных панелях значил бы разное.

In [ ]:
SNAP_T_RANGE = None      # (t0, t1) мс — только снимки из окна, напр. (1500, 1950); None — все
MAX_PANELS = 12          # больше панелей — берутся равномерно

snaps = [sn for sn in run.snapshots()
         if SNAP_T_RANGE is None or SNAP_T_RANGE[0] <= sn.t_ms <= SNAP_T_RANGE[1]]
if snaps:
    vals = [sn.grid("e_state", V_NAME) for sn in snaps]
    vmin, vmax = min(v.min() for v in vals), max(v.max() for v in vals)
    pick = np.unique(np.linspace(0, len(snaps) - 1, min(len(snaps), MAX_PANELS)).round().astype(int))
    ncol = min(len(pick), 4)
    nrow = int(np.ceil(len(pick) / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 3.8 * nrow),
                            constrained_layout=True, squeeze=False)
    for ax, i in zip(axs.flat, pick):
        sn, v = snaps[i], vals[i]
        im = ax.imshow(v, origin="lower", extent=EXTENT, cmap=style.seq, vmin=vmin, vmax=vmax,
                       interpolation="bilinear", aspect="equal")
        ax.set_title(f"t = {sn.t_ms:g} мс")
        ax.grid(False)
    for ax in list(axs.flat)[len(pick):]:
        ax.axis("off")
    fig.colorbar(im, ax=axs, shrink=0.8, label=f"{V_NAME}, {V_UNIT}")
    save(fig, "snapshots_potential")
else:
    print("снимков нет (output.snapshot_times_ms)")

## 4. Механика в снимке

Деформированная механическая сетка (перемещения в масштабе 1:1), цвет — растяжение волокна
относительно преднагрузки: синий — укорочение, красный — удлинение, серый — как при
преднагрузке. Справа — действующее активное напряжение.

In [ ]:
SNAP_T = None      # момент снимка, мс; None — снимок с наибольшим активным напряжением

if snaps:
    sn = (max(snaps, key=lambda q: q["m_t_act"].max()) if SNAP_T is None
          else run.snapshot(SNAP_T))
    ny, nx = M["ny"], M["nx"]
    xy = sn["m_coords"].reshape(ny + 1, nx + 1, 2)
    u = sn["m_u"].reshape(ny + 1, nx + 1, 2)
    X, Y = xy[..., 0] + u[..., 0], xy[..., 1] + u[..., 1]
    lam = sn.grid("m_lambda_f")
    t_act = sn.grid("m_t_act")
    span = max(abs(lam.max() - PRELOAD), abs(lam.min() - PRELOAD), 1e-6)
    norm = TwoSlopeNorm(vcenter=PRELOAD, vmin=PRELOAD - span, vmax=PRELOAD + span)

    fig, axs = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
    pc = axs[0].pcolormesh(X, Y, lam, cmap=style.div, norm=norm, edgecolors=style.surface, linewidth=0.4)
    axs[0].set_title(f"λ_f на деформированной сетке, t = {sn.t_ms:g} мс")
    fig.colorbar(pc, ax=axs[0], shrink=0.85, label="λ_f")
    pc2 = axs[1].pcolormesh(X, Y, t_act, cmap=style.seq, edgecolors=style.surface, linewidth=0.4)
    axs[1].set_title("Действующее активное напряжение")
    fig.colorbar(pc2, ax=axs[1], shrink=0.85, label="кПа")
    for ax in axs:
        ax.set_aspect("equal")
        ax.set_xlabel("x, мм")
        ax.set_ylabel("y, мм")
        ax.grid(False)
    save(fig, f"mechanics_t{sn.t_ms:g}")

## 5. Анимация снимков

Встроенный плеер в ноутбуке. Для плавной анимации нужно больше снимков
(`output.snapshot_times_ms`) — или ParaView по XDMF.

In [ ]:
from IPython.display import HTML
from matplotlib import animation

ANIMATE = len(snaps) > 1
if ANIMATE:
    fig, ax = plt.subplots(figsize=(5.5, 4.5), constrained_layout=True)
    im = ax.imshow(vals[0], origin="lower", extent=EXTENT, cmap=style.seq, vmin=vmin, vmax=vmax,
                   interpolation="bilinear", aspect="equal")
    ax.grid(False)
    fig.colorbar(im, ax=ax, shrink=0.85, label=f"{V_NAME}, {V_UNIT}")
    title = ax.set_title("")

    def frame(i):
        im.set_data(vals[i])
        title.set_text(f"t = {snaps[i].t_ms:g} мс")
        return im, title

    anim = animation.FuncAnimation(fig, frame, frames=len(snaps), interval=400, blit=False)
    plt.close(fig)
    display(HTML(anim.to_jshtml()))

## 6. APD по регионам и реституция

Распределение APD в каждом регионе (0 — базовая ткань). Реституция (APD следующего удара
против диастолического интервала) — если ударов больше одного.

In [ ]:
if maps is not None and maps.n_beats:
    names = {0: "базовая ткань", **{i + 1: r.get("name") or f"регион {i + 1}"
                                    for i, r in enumerate(run.config.get("regions", []))}}
    regs = [r for r in np.unique(maps.region)]
    data = [maps.apd[maps.region == r, B] for r in regs]
    data = [d[np.isfinite(d)] for d in data]
    ncols = 2 if maps.n_beats > 1 else 1
    fig, axs = plt.subplots(1, ncols, figsize=(6 * ncols, 4), constrained_layout=True, squeeze=False)
    ax = axs[0, 0]
    bp = ax.boxplot(data, patch_artist=True, widths=0.5, medianprops={"color": style.ink})
    for patch, c in zip(bp["boxes"], style.series):
        patch.set_facecolor(c)
        patch.set_alpha(0.55)
    ax.set_xticks(range(1, len(regs) + 1), [names.get(int(r), r) for r in regs])
    ax.set_ylabel("мс")
    ax.set_title(f"APD по регионам, удар {B + 1}")
    if maps.n_beats > 1:
        rest = bm.restitution(maps.act, maps.repol)
        ax = axs[0, 1]
        ax.scatter(rest["di_ms"], rest["apd_ms"], s=10, color=style.series[0], alpha=0.6)
        ax.set_xlabel("диастолический интервал, мс")
        ax.set_ylabel("APD, мс")
        ax.set_title("Реституция APD")
    save(fig, "apd_regions")

## 7. Серия

Биомаркер против параметра серии; если параметров два — линия на каждое значение второго
(с легендой). Рисунок — в `<серия>/figures/`.

In [ ]:
SWEEP_DIR = RUNS / "nb_quick_sweep"  # ночная серия: RUNS / "night"
SWEEP_BEAT = -1                      # удар для метрик: -1 — последний
METRIC = "APD регион 1, мс"   # или ключ из bm.mechanics_summary / activation_summary, напр. "peak_t_act_kpa"


def metrics(r):
    out = bm.mechanics_summary(r.series())
    try:
        mp = r.activation()
        b = SWEEP_BEAT % mp.n_beats
        out.update(bm.activation_summary(mp.act[:, b]))
        for reg, st in bm.apd_summary(mp.apd[:, b], mp.region)["by_region"].items():
            out[f"APD регион {reg}, мс"] = st["mean"]
    except FileNotFoundError:
        pass
    return out


tab = None
if not (SWEEP_DIR / "sweep.json").exists():
    print(f"серии в {SWEEP_DIR.relative_to(ROOT)} нет — запустите её в 01_run.ipynb (раздел 6)")
else:
    import pandas as pd
    sweep = open_sweep(SWEEP_DIR)
    params = list(sweep.parameters)
    done = sum(p["status"] == "finished" for p in sweep.points)
    print(f"посчитано точек: {done} из {len(sweep.points)}")
    tab = pd.DataFrame(sweep.table(metrics))
    if tab.empty:
        tab = None
        print("считать нечего — запустите серию в 01_run.ipynb (раздел 6, RUN_SWEEP или RUN_SWEEP_MPI)")
    elif METRIC not in tab:
        print(f"метрики {METRIC!r} нет; есть: {[c for c in tab.columns if c not in params and c != 'name']}")
        tab = None

if tab is not None:
    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    short = [p.split(".")[-1].removeprefix("cell:") for p in params]
    if sweep.index.get("mode") == "zip" and len(params) > 1:
        # точки — согласованные наборы (стадии): столбцы с подписью набора
        labels = ["\n".join(f"{n}={row[p]:g}" for n, p in zip(short, params)) for _, row in tab.iterrows()]
        ax.bar(range(len(tab)), tab[METRIC], color=style.series[0], width=0.6)
        ax.set_xticks(range(len(tab)), labels, fontsize=8)
        for i, v in enumerate(tab[METRIC]):
            ax.annotate(f"{v:.1f}", (i, v), xytext=(0, 3), textcoords="offset points",
                        ha="center", color=style.ink, fontsize=9)
        ax.grid(axis="x", visible=False)
    elif len(params) == 1:
        ax.plot(tab[params[0]], tab[METRIC], marker="o")
    else:
        for (val, grp), c in zip(tab.groupby(params[1]), style.series):
            ax.plot(grp[params[0]], grp[METRIC], marker="o", color=c,
                    markeredgecolor=style.surface, markeredgewidth=2,
                    label=f"{params[1].split('.')[-1]} = {val}")
        ax.legend()
    if not (sweep.index.get("mode") == "zip" and len(params) > 1):
        ax.set_xlabel(short[0])
    ax.set_ylabel(METRIC)
    ax.margins(x=0.08)
    ax.set_title(f"Серия {SWEEP_DIR.name}: {METRIC}")
    if SAVE:
        (SWEEP_DIR / "figures").mkdir(exist_ok=True)
        fig.savefig(SWEEP_DIR / "figures" / f"{METRIC.split(',')[0].replace(' ', '_')}.png",
                    dpi=160, bbox_inches="tight")